# Perseptron v2 - 07 Optimized Kaggle Submission Generation

This notebook creates full-customer Kaggle submission files for the three recommendation models: `tabular_only`, `image_history`, and `late_fusion`.

The notebook uses a candidate-generation plus reranking pipeline. All customers from `sample_submission.csv` are preserved, while each model scores a compact candidate set instead of the whole article catalog.

Outputs:

- `submission_tabular_only.csv`
- `submission_image_history.csv`
- `submission_late_fusion.csv`
- `submission_generation_summary.md`

The image-only CNN is not used here because it is a classification/explainability baseline, not a customer-level recommender.

## Environment and settings

This cell discovers Kaggle inputs, sets full-run submission parameters, and keeps all text ASCII to avoid notebook rendering issues.

In [ ]:
from pathlib import Path
import gc
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
import torch
from torch import nn
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

IS_KAGGLE = Path('/kaggle').exists()
WORK_DIR = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()
INPUT_ROOTS = [Path('/kaggle/input')] if IS_KAGGLE else [Path.cwd(), Path.cwd() / 'artifacts' / 'final', Path.cwd() / 'data']

# Direct full run. Change to True only for local/debug checks.
SMOKE_RUN = False
SMOKE_CUSTOMERS = 1000

TOP_K = 12
POPULAR_CANDIDATES = 80
LAST_7D_CANDIDATES = 80
LAST_30D_CANDIDATES = 80
GLOBAL_CANDIDATES = 80
RECENT_PURCHASE_CANDIDATES = 24
SIMILAR_PRODUCT_TYPE_CANDIDATES = 16
SIMILAR_GARMENT_GROUP_CANDIDATES = 16
CUSTOMER_CANDIDATE_LIMIT = 180
INFERENCE_CUSTOMER_BATCH_SIZE = 2048
INFERENCE_SCORE_BATCH_SIZE = 8192
VISUAL_FEATURE_BATCH_SIZE = 32768
PROFILE_BUILD_BATCH_SIZE = 50000

MODEL_NAMES = ['tabular_only', 'image_history', 'late_fusion']
MODEL_BASENAMES = {
    'tabular_only': ['tabular_only.pt', 'tabular_only_fold0.pt'],
    'image_history': ['image_history.pt', 'image_history_fold0.pt'],
    'late_fusion': ['late_fusion.pt', 'late_fusion_fold0.pt'],
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
START_TIME = time.time()

def log(message):
    print(f'[{time.strftime("%H:%M:%S")}] {message}', flush=True)

def find_file(filename, required=True):
    candidates = []
    for root in INPUT_ROOTS:
        if not root.exists():
            continue
        candidates.extend(root.rglob(filename))
    if not candidates and Path(filename).exists():
        candidates.append(Path(filename))
    if not candidates:
        if required:
            raise FileNotFoundError(f'{filename} not found. Check Kaggle Add Data inputs.')
        return None
    return sorted(candidates, key=lambda path: (len(str(path)), str(path)))[0]

def find_first_file(filenames, required=True):
    for filename in filenames:
        path = find_file(filename, required=False)
        if path is not None:
            return path
    if required:
        raise FileNotFoundError(f'None of these files were found: {filenames}')
    return None

TRANSACTIONS_PATH = find_file('transactions_train.csv')
CUSTOMERS_PATH = find_file('customers.csv')
ARTICLES_PATH = find_file('articles.csv')
SAMPLE_SUBMISSION_PATH = find_file('sample_submission.csv')
EMBEDDINGS_PATH = find_file('article_image_embeddings_popular.npy')
EMBEDDING_IDS_PATH = find_file('article_image_embedding_ids_popular.csv')
MODEL_PATHS = {name: find_first_file(filenames) for name, filenames in MODEL_BASENAMES.items()}

print('work_dir:', WORK_DIR)
print('device:', DEVICE)
if torch.cuda.is_available():
    print('cuda devices:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('transactions:', TRANSACTIONS_PATH)
print('customers:', CUSTOMERS_PATH)
print('articles:', ARTICLES_PATH)
print('sample_submission:', SAMPLE_SUBMISSION_PATH)
print('embeddings:', EMBEDDINGS_PATH)
print('embedding ids:', EMBEDDING_IDS_PATH)
print('models:', MODEL_PATHS)

## Model classes and encoders

These definitions rebuild the checkpoint architectures and encode tabular metadata without importing project source files.

In [ ]:
def embedding_dim(size: int) -> int:
    return min(50, max(4, int(size**0.25 * 8)))

class TabularOnlyMLP(nn.Module):
    def __init__(self, numeric_dim: int, category_sizes: list[int]):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(size, embedding_dim(size)) for size in category_sizes])
        cat_dim = sum(embedding.embedding_dim for embedding in self.embeddings)
        self.net = nn.Sequential(
            nn.Linear(numeric_dim + cat_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, 1),
        )

    def forward(self, numeric, categorical):
        embedded = [emb(categorical[:, idx]) for idx, emb in enumerate(self.embeddings)]
        return self.net(torch.cat([numeric, *embedded], dim=1)).squeeze(1)

class ImageHistoryMLP(nn.Module):
    def __init__(self, image_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(image_dim * 2 + 2, 512), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, 1),
        )

    def forward(self, article_embedding, profile_embedding, visual_similarity, visual_history_count):
        visual_extra = torch.stack([visual_similarity, visual_history_count], dim=1)
        return self.net(torch.cat([article_embedding, profile_embedding, visual_extra], dim=1)).squeeze(1)

class MultimodalLateFusion(nn.Module):
    def __init__(self, numeric_dim: int, category_sizes: list[int], image_dim: int):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(size, embedding_dim(size)) for size in category_sizes])
        cat_dim = sum(embedding.embedding_dim for embedding in self.embeddings)
        self.tabular_branch = nn.Sequential(
            nn.Linear(numeric_dim + cat_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.visual_branch = nn.Sequential(
            nn.Linear(image_dim * 2 + 2, 512), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(512, 128), nn.ReLU(),
        )
        self.fusion_head = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, 1))

    def forward(self, numeric, categorical, article_embedding, profile_embedding, visual_similarity, visual_history_count):
        embedded = [emb(categorical[:, idx]) for idx, emb in enumerate(self.embeddings)]
        tabular = self.tabular_branch(torch.cat([numeric, *embedded], dim=1))
        visual_extra = torch.stack([visual_similarity, visual_history_count], dim=1)
        visual = self.visual_branch(torch.cat([article_embedding, profile_embedding, visual_extra], dim=1))
        return self.fusion_head(torch.cat([tabular, visual], dim=1)).squeeze(1)

def build_model(model_name: str, metadata: dict, image_dim: int):
    category_sizes = [len(metadata['category_maps'][column]) for column in metadata['categorical_features']]
    numeric_dim = len(metadata['numeric_features'])
    if model_name == 'tabular_only':
        return TabularOnlyMLP(numeric_dim, category_sizes)
    if model_name == 'image_history':
        return ImageHistoryMLP(image_dim)
    if model_name == 'late_fusion':
        return MultimodalLateFusion(numeric_dim, category_sizes, image_dim)
    raise ValueError(f'Unknown model: {model_name}')

def load_checkpoint_model(path: Path, device: torch.device):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model = build_model(checkpoint['model_name'], checkpoint['metadata'], checkpoint['image_dim']).to(device)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return checkpoint, model

def encode_tabular(frame: pd.DataFrame, metadata: dict):
    numeric_columns = []
    for column in metadata['numeric_features']:
        values = pd.to_numeric(frame[column], errors='coerce').fillna(metadata['numeric_mean'][column]).astype('float32')
        values = (values - metadata['numeric_mean'][column]) / metadata['numeric_std'][column]
        numeric_columns.append(values.to_numpy(dtype='float32'))
    numeric = np.stack(numeric_columns, axis=1).astype('float32')

    categorical_columns = []
    for column in metadata['categorical_features']:
        mapping = metadata['category_maps'][column]
        values = frame[column].fillna('UNKNOWN').astype(str).map(mapping).fillna(0).astype('int64')
        categorical_columns.append(values.to_numpy(dtype='int64'))
    categorical = np.stack(categorical_columns, axis=1).astype('int64')
    return numeric, categorical

## Load data, checkpoints, and visual profiles

This cell loads full H&M data, the embedding cache, and the three final recommender checkpoints. It then builds customer visual profile sums used by image-history and late-fusion models.

In [ ]:
def normalize_article_id_series(series):
    return series.astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(10)

log('Loading raw H&M tables...')
transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    dtype={'customer_id': 'string', 'article_id': 'string'},
    usecols=['customer_id', 'article_id', 't_dat'],
)
transactions['article_id'] = normalize_article_id_series(transactions['article_id'])
transactions['customer_id'] = transactions['customer_id'].astype(str)
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

customers = pd.read_csv(CUSTOMERS_PATH, dtype={'customer_id': 'string'})
customers['customer_id'] = customers['customer_id'].astype(str)
articles = pd.read_csv(ARTICLES_PATH, dtype={'article_id': 'string'})
articles['article_id'] = normalize_article_id_series(articles['article_id'])
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH, dtype={'customer_id': 'string'})
sample_submission['customer_id'] = sample_submission['customer_id'].astype(str)

if SMOKE_RUN:
    sample_submission = sample_submission.head(SMOKE_CUSTOMERS).copy()
    log(f'Smoke mode active: {len(sample_submission):,} customers will be processed.')
else:
    log(f'Full mode active: {len(sample_submission):,} customers will be processed.')

log('Loading embedding cache...')
embeddings = np.load(EMBEDDINGS_PATH, mmap_mode=None).astype('float32')
embedding_ids = pd.read_csv(EMBEDDING_IDS_PATH, dtype={'article_id': 'string'})
article_ids = normalize_article_id_series(embedding_ids['article_id']).tolist()
article_to_index = {article_id: index for index, article_id in enumerate(article_ids)}
embedded_article_set = set(article_to_index)

log('Building customer visual profile arrays...')
embedded_history = transactions[transactions['article_id'].isin(embedded_article_set)][['customer_id', 'article_id']].copy()
customer_ids_with_profile = embedded_history['customer_id'].drop_duplicates().tolist()
customer_to_index = {customer_id: index for index, customer_id in enumerate(customer_ids_with_profile)}
customer_indices = embedded_history['customer_id'].map(customer_to_index).to_numpy(dtype='int64')
article_indices = embedded_history['article_id'].map(article_to_index).to_numpy(dtype='int64')
profile_sums = np.zeros((len(customer_ids_with_profile), embeddings.shape[1]), dtype='float32')
for start in tqdm(range(0, len(article_indices), PROFILE_BUILD_BATCH_SIZE), desc='Profile sum batches'):
    end = min(start + PROFILE_BUILD_BATCH_SIZE, len(article_indices))
    np.add.at(profile_sums, customer_indices[start:end], embeddings[article_indices[start:end]])
profile_counts = np.bincount(customer_indices, minlength=len(customer_ids_with_profile)).astype('float32')
del customer_indices, article_indices, embedded_history
gc.collect()

log('Loading checkpoints...')
models = {}
for name, path in MODEL_PATHS.items():
    checkpoint, model = load_checkpoint_model(path, DEVICE)
    models[name] = (checkpoint, model)
    log(f'Model loaded: {name} -> {path.name}')

log('Moving lookup tensors to GPU if available...')
ARTICLE_TENSOR = torch.tensor(embeddings, dtype=torch.float32, device=DEVICE) if DEVICE.type == 'cuda' else None
PROFILE_SUM_TENSOR = torch.tensor(profile_sums, dtype=torch.float32, device=DEVICE) if DEVICE.type == 'cuda' else None
PROFILE_COUNT_TENSOR = torch.tensor(profile_counts, dtype=torch.float32, device=DEVICE) if DEVICE.type == 'cuda' else None
if DEVICE.type == 'cuda':
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    log(f'GPU lookup tensors ready | allocated={allocated:.2f}GB | reserved={reserved:.2f}GB')

print('transactions:', transactions.shape)
print('customers:', customers.shape)
print('articles:', articles.shape)
print('sample_submission:', sample_submission.shape)
print('embeddings:', embeddings.shape)
print('customer profiles:', profile_sums.shape)

## Candidate generation

This cell creates compact candidate pools: recent time-aware popularity, customer recent purchases, and metadata-similar items by product type and garment group.

In [ ]:
def top_articles_for_window(transactions_frame, start_date, limit, article_filter=None):
    frame = transactions_frame
    if start_date is not None:
        frame = frame[frame['t_dat'] >= start_date]
    if article_filter is not None:
        frame = frame[frame['article_id'].isin(article_filter)]
    return frame['article_id'].value_counts().head(limit).index.astype(str).tolist()

def build_popular_candidates(transactions_frame, article_filter):
    log('Building time-aware popularity candidates...')
    max_date = transactions_frame['t_dat'].max()
    last_7d_start = max_date - pd.Timedelta(days=7)
    last_30d_start = max_date - pd.Timedelta(days=30)
    last_7d = top_articles_for_window(transactions_frame, last_7d_start, LAST_7D_CANDIDATES, article_filter)
    last_30d = top_articles_for_window(transactions_frame, last_30d_start, LAST_30D_CANDIDATES, article_filter)
    global_popular = top_articles_for_window(transactions_frame, None, GLOBAL_CANDIDATES, article_filter)
    candidates = list(dict.fromkeys(last_7d + last_30d + global_popular))[:POPULAR_CANDIDATES]
    log(f'Last-7d candidates: {len(last_7d):,}')
    log(f'Last-30d candidates: {len(last_30d):,}')
    log(f'Global candidates: {len(global_popular):,}')
    log(f'Blended popular candidates: {len(candidates):,}')
    return candidates

def build_customer_recent_candidates(transactions_frame, article_filter):
    log('Building customer recent-purchase candidates...')
    recent_transactions = transactions_frame[transactions_frame['article_id'].isin(article_filter)].copy()
    recent_transactions = recent_transactions.sort_values(['customer_id', 't_dat'], ascending=[True, False])
    recent_candidates = (
        recent_transactions.groupby('customer_id')['article_id']
        .apply(lambda values: list(dict.fromkeys(values.tolist()))[:RECENT_PURCHASE_CANDIDATES])
        .to_dict()
    )
    log(f'Customers with recent candidates: {len(recent_candidates):,}')
    return recent_candidates

def build_metadata_similarity_candidates(transactions_frame, articles_frame, article_filter):
    log('Building metadata-similar candidates...')
    embedded_articles_df = articles_frame[articles_frame['article_id'].isin(article_filter)].copy()
    embedded_transactions = transactions_frame[transactions_frame['article_id'].isin(article_filter)].copy()
    article_popularity = embedded_transactions['article_id'].value_counts()
    embedded_articles_df['popularity'] = embedded_articles_df['article_id'].map(article_popularity).fillna(0)

    product_type_top = (
        embedded_articles_df.sort_values(['product_type_no', 'popularity'], ascending=[True, False])
        .groupby('product_type_no')['article_id']
        .apply(lambda values: values.head(SIMILAR_PRODUCT_TYPE_CANDIDATES).tolist())
        .to_dict()
    )
    garment_group_top = (
        embedded_articles_df.sort_values(['garment_group_no', 'popularity'], ascending=[True, False])
        .groupby('garment_group_no')['article_id']
        .apply(lambda values: values.head(SIMILAR_GARMENT_GROUP_CANDIDATES).tolist())
        .to_dict()
    )
    article_meta = embedded_articles_df.set_index('article_id')[['product_type_no', 'garment_group_no']].to_dict('index')
    recent_transactions = embedded_transactions.sort_values(['customer_id', 't_dat'], ascending=[True, False])

    customer_candidates = {}
    for customer_id, values in tqdm(recent_transactions.groupby('customer_id')['article_id'], desc='Metadata candidates'):
        candidates = []
        for article_id in list(dict.fromkeys(values.tolist()))[:RECENT_PURCHASE_CANDIDATES]:
            meta = article_meta.get(article_id)
            if not meta:
                continue
            candidates.extend(product_type_top.get(meta['product_type_no'], []))
            candidates.extend(garment_group_top.get(meta['garment_group_no'], []))
            if len(candidates) >= SIMILAR_PRODUCT_TYPE_CANDIDATES + SIMILAR_GARMENT_GROUP_CANDIDATES:
                break
        customer_candidates[customer_id] = list(dict.fromkeys(candidates))
    log(f'Customers with metadata-similar candidates: {len(customer_candidates):,}')
    return customer_candidates

popular_articles = build_popular_candidates(transactions, embedded_article_set)
customer_recent_candidates = build_customer_recent_candidates(transactions, embedded_article_set)
customer_similar_candidates = build_metadata_similarity_candidates(transactions, articles, embedded_article_set)
log('Candidate pools ready.')

## Batched scoring helpers

These helpers build a candidate grid per customer batch, compute visual scalar features, and score the same candidate grid with each model.

In [ ]:
def make_visual_batch_from_indices(article_idx, customer_idx, pair_counts):
    if DEVICE.type == 'cuda':
        article_b = torch.as_tensor(article_idx, dtype=torch.long, device=DEVICE)
        customer_b = torch.as_tensor(customer_idx, dtype=torch.long, device=DEVICE)
        pair_b = torch.as_tensor(pair_counts, dtype=torch.float32, device=DEVICE)
        article_emb = ARTICLE_TENSOR[article_b]
        sums = PROFILE_SUM_TENSOR[customer_b] - pair_b.unsqueeze(1) * article_emb
        counts = PROFILE_COUNT_TENSOR[customer_b] - pair_b
        profiles = sums / torch.clamp(counts, min=1).unsqueeze(1)
        profiles = torch.nn.functional.normalize(profiles, p=2, dim=1)
        profiles = torch.where(counts.unsqueeze(1) > 0, profiles, torch.zeros_like(profiles))
        visual_similarity = torch.sum(profiles * article_emb, dim=1)
        return article_emb, profiles, visual_similarity, counts

    article_emb_np = embeddings[article_idx]
    sums_np = profile_sums[customer_idx] - pair_counts[:, None] * article_emb_np
    counts_np = profile_counts[customer_idx] - pair_counts
    profiles_np = sums_np / np.maximum(counts_np, 1.0)[:, None]
    norms = np.linalg.norm(profiles_np, axis=1, keepdims=True)
    profiles_np = profiles_np / np.maximum(norms, 1e-8)
    profiles_np[counts_np <= 0] = 0
    visual_similarity_np = np.sum(profiles_np * article_emb_np, axis=1).astype('float32')
    return (
        torch.tensor(article_emb_np, dtype=torch.float32, device=DEVICE),
        torch.tensor(profiles_np, dtype=torch.float32, device=DEVICE),
        torch.tensor(visual_similarity_np, dtype=torch.float32, device=DEVICE),
        torch.tensor(counts_np, dtype=torch.float32, device=DEVICE),
    )

def add_visual_scalar_features(grid):
    article_idx = grid['article_index'].to_numpy(dtype='int64')
    customer_idx = grid['customer_index'].to_numpy(dtype='int64')
    pair_counts = grid['pair_purchase_count'].to_numpy(dtype='float32')
    visual_similarity = np.zeros(len(grid), dtype='float32')
    visual_history_count = np.zeros(len(grid), dtype='float32')
    with torch.no_grad():
        for start in range(0, len(grid), VISUAL_FEATURE_BATCH_SIZE):
            end = min(start + VISUAL_FEATURE_BATCH_SIZE, len(grid))
            _, _, sim, hist = make_visual_batch_from_indices(article_idx[start:end], customer_idx[start:end], pair_counts[start:end])
            visual_similarity[start:end] = sim.detach().cpu().numpy()
            visual_history_count[start:end] = hist.detach().cpu().numpy()
    grid['visual_similarity'] = visual_similarity
    grid['visual_history_count'] = visual_history_count
    return grid

def score_model_for_grid(model_name, checkpoint, model, grid):
    article_idx = grid['article_index'].to_numpy(dtype='int64')
    customer_idx = grid['customer_index'].to_numpy(dtype='int64')
    pair_counts = grid['pair_purchase_count'].to_numpy(dtype='float32')

    if model_name in {'tabular_only', 'late_fusion'}:
        numeric, categorical = encode_tabular(grid, checkpoint['metadata'])
    else:
        numeric = categorical = None

    scores = []
    model.eval()
    with torch.no_grad():
        for start in range(0, len(grid), INFERENCE_SCORE_BATCH_SIZE):
            end = min(start + INFERENCE_SCORE_BATCH_SIZE, len(grid))
            if model_name == 'tabular_only':
                numeric_b = torch.tensor(numeric[start:end], dtype=torch.float32, device=DEVICE)
                categorical_b = torch.tensor(categorical[start:end], dtype=torch.long, device=DEVICE)
                logits = model(numeric_b, categorical_b)
            elif model_name == 'image_history':
                article_emb, profiles, sim, hist = make_visual_batch_from_indices(
                    article_idx[start:end], customer_idx[start:end], pair_counts[start:end]
                )
                logits = model(article_emb, profiles, sim, hist)
            elif model_name == 'late_fusion':
                numeric_b = torch.tensor(numeric[start:end], dtype=torch.float32, device=DEVICE)
                categorical_b = torch.tensor(categorical[start:end], dtype=torch.long, device=DEVICE)
                article_emb, profiles, sim, hist = make_visual_batch_from_indices(
                    article_idx[start:end], customer_idx[start:end], pair_counts[start:end]
                )
                logits = model(numeric_b, categorical_b, article_emb, profiles, sim, hist)
            else:
                raise ValueError(model_name)
            scores.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(scores).astype('float32') if scores else np.array([], dtype='float32')

def top_predictions_from_scores(grid, scores):
    scored = grid[['customer_id', 'article_id']].copy()
    scored['score'] = scores
    top = (
        scored.sort_values(['customer_id', 'score'], ascending=[True, False])
        .groupby('customer_id')['article_id']
        .apply(lambda values: ' '.join(list(values.head(TOP_K))))
    )
    return top.to_dict()

def pad_prediction(prediction, fallback_items):
    items = str(prediction).split() if prediction else []
    seen = set(items)
    for item in fallback_items:
        if len(items) >= TOP_K:
            break
        if item not in seen:
            seen.add(item)
            items.append(item)
    return ' '.join(items[:TOP_K])

## Create full submission files

This cell keeps all customers from `sample_submission.csv`, fills cold-start customers with popularity fallback, and replaces predictions for scoreable customers after model reranking.

In [ ]:
fallback_prediction = ' '.join(popular_articles[:TOP_K])
submission_customers = sample_submission['customer_id'].astype(str).tolist()
customer_position = {customer_id: index for index, customer_id in enumerate(submission_customers)}
scoreable_customers = [customer_id for customer_id in submission_customers if customer_id in customer_to_index]
scoreable_candidates = [article_id for article_id in popular_articles if article_id in article_to_index]

log(f'Submission customers: {len(submission_customers):,}')
log(f'Scoreable customers: {len(scoreable_customers):,}')
log(f'Fallback customers: {len(submission_customers) - len(scoreable_customers):,}')
log(f'Global candidate articles: {len(scoreable_candidates):,}')

log('Preparing shared article/customer metadata for scoring...')
all_candidate_articles = set(scoreable_candidates)
for customer_id in tqdm(scoreable_customers, desc='Collecting all candidates'):
    all_candidate_articles.update(customer_recent_candidates.get(customer_id, []))
    all_candidate_articles.update(customer_similar_candidates.get(customer_id, []))
all_candidate_articles = [article_id for article_id in all_candidate_articles if article_id in article_to_index]
log(f'Total unique candidate articles for scoring: {len(all_candidate_articles):,}')

articles_meta = articles[articles['article_id'].isin(all_candidate_articles)].copy()
customer_meta = customers[customers['customer_id'].isin(scoreable_customers)].copy()
pair_counts = (
    transactions[
        transactions['customer_id'].isin(scoreable_customers)
        & transactions['article_id'].isin(all_candidate_articles)
    ]
    .groupby(['customer_id', 'article_id'])
    .size()
    .reset_index(name='pair_purchase_count')
)
log(f'Historical customer/article pair counts: {len(pair_counts):,}')

prediction_arrays = {name: np.full(len(submission_customers), fallback_prediction, dtype=object) for name in MODEL_NAMES}
scored_customer_counts = {name: 0 for name in MODEL_NAMES}
candidate_counts = []

for batch_id, start in enumerate(tqdm(range(0, len(scoreable_customers), INFERENCE_CUSTOMER_BATCH_SIZE), desc='Submission customer batches'), start=1):
    batch_customers = scoreable_customers[start:start + INFERENCE_CUSTOMER_BATCH_SIZE]
    rows = []
    for customer_id in batch_customers:
        customer_candidates = []
        customer_candidates.extend(customer_recent_candidates.get(customer_id, []))
        customer_candidates.extend(customer_similar_candidates.get(customer_id, []))
        customer_candidates.extend(scoreable_candidates)
        customer_candidates = [
            article_id for article_id in dict.fromkeys(customer_candidates)
            if article_id in article_to_index
        ][:CUSTOMER_CANDIDATE_LIMIT]
        if len(customer_candidates) < TOP_K:
            customer_candidates = list(dict.fromkeys(customer_candidates + scoreable_candidates))[:CUSTOMER_CANDIDATE_LIMIT]
        candidate_counts.append(len(customer_candidates))
        for article_id in customer_candidates:
            rows.append((customer_id, article_id))

    if not rows:
        continue

    grid = pd.DataFrame(rows, columns=['customer_id', 'article_id'])
    grid = grid.merge(customer_meta, on='customer_id', how='left')
    grid = grid.merge(articles_meta, on='article_id', how='left')
    grid = grid.merge(pair_counts, on=['customer_id', 'article_id'], how='left')
    grid['pair_purchase_count'] = grid['pair_purchase_count'].fillna(0).astype('float32')
    grid['article_index'] = grid['article_id'].map(article_to_index).astype('int64')
    grid['customer_index'] = grid['customer_id'].map(customer_to_index).astype('int64')
    grid = add_visual_scalar_features(grid)

    for model_name in MODEL_NAMES:
        checkpoint, model = models[model_name]
        scores = score_model_for_grid(model_name, checkpoint, model, grid)
        top_predictions = top_predictions_from_scores(grid, scores)
        for customer_id, prediction in top_predictions.items():
            prediction_arrays[model_name][customer_position[customer_id]] = pad_prediction(prediction, scoreable_candidates)
            scored_customer_counts[model_name] += 1

    if batch_id % 10 == 0 or start + len(batch_customers) >= len(scoreable_customers):
        elapsed_hours = (time.time() - START_TIME) / 3600
        log(f'Batch {batch_id:,} complete | customers={start + len(batch_customers):,}/{len(scoreable_customers):,} | elapsed={elapsed_hours:.2f} h')

    del grid, rows
    gc.collect()

summary_lines = [
    '# Submission Generation Summary',
    '',
    f'- Smoke run: {SMOKE_RUN}',
    f'- Customers in output: {len(submission_customers):,}',
    f'- Scoreable customers: {len(scoreable_customers):,}',
    f'- Fallback customers before scoring: {len(submission_customers) - len(scoreable_customers):,}',
    f'- Top-k: {TOP_K}',
    f'- Customer candidate limit: {CUSTOMER_CANDIDATE_LIMIT}',
    f'- Inference customer batch size: {INFERENCE_CUSTOMER_BATCH_SIZE}',
    f'- Inference score batch size: {INFERENCE_SCORE_BATCH_SIZE}',
    f'- Mean candidate count: {float(np.mean(candidate_counts)) if candidate_counts else 0:.2f}',
    f'- Runtime hours: {(time.time() - START_TIME) / 3600:.2f}',
    f'- Device: {DEVICE}',
    '',
    '| model | output | rows | scored_customers | fallback_rows | unique_predictions | valid_top12 |',
    '| --- | --- | ---: | ---: | ---: | ---: | --- |',
]

for model_name in MODEL_NAMES:
    output = pd.DataFrame({'customer_id': submission_customers, 'prediction': prediction_arrays[model_name]})
    output['prediction'] = output['prediction'].map(lambda value: pad_prediction(value, scoreable_candidates))
    output_path = WORK_DIR / f'submission_{model_name}.csv'
    output.to_csv(output_path, index=False)
    prediction_lengths = output['prediction'].astype(str).str.split().map(len)
    valid_top12 = bool(prediction_lengths.eq(TOP_K).all())
    if list(output.columns) != ['customer_id', 'prediction']:
        raise AssertionError(f'{model_name}: invalid columns')
    if len(output) != len(sample_submission):
        raise AssertionError(f'{model_name}: row count does not match sample submission')
    if not valid_top12:
        raise AssertionError(f'{model_name}: some predictions do not contain {TOP_K} articles')
    unique_predictions = int(output['prediction'].nunique())
    fallback_rows = int((output['prediction'] == fallback_prediction).sum())
    summary_lines.append(
        f'| {model_name} | `{output_path.name}` | {len(output):,} | {scored_customer_counts[model_name]:,} | {fallback_rows:,} | {unique_predictions:,} | {valid_top12} |'
    )
    log(f'Saved: {output_path}')

summary_path = WORK_DIR / 'submission_generation_summary.md'
summary_path.write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
print('\n'.join(summary_lines))
log(f'Summary saved: {summary_path}')

## Submit commands

Do not submit automatically from this notebook. After full output validation, submit manually in this order:

```bash
kaggle competitions submit -c h-and-m-personalized-fashion-recommendations -f /kaggle/working/submission_late_fusion.csv -m "perseptron v2 late_fusion optimized full"
kaggle competitions submit -c h-and-m-personalized-fashion-recommendations -f /kaggle/working/submission_tabular_only.csv -m "perseptron v2 tabular_only optimized full"
kaggle competitions submit -c h-and-m-personalized-fashion-recommendations -f /kaggle/working/submission_image_history.csv -m "perseptron v2 image_history optimized full"
```